# KATL V19-B — Dense v11 Settlement + Empirical Modal Bucket

This notebook implements V19-B only: the v11 settlement remaining-warmup ridge stack, a training-only 3% feature-missingness gate, cross-fitted ridge residuals, and station/month empirical modal-bucket selection. It does not train the V19-C ordinal classifier and does not export a production bundle.


In [ ]:
from pathlib import Path
import os
import sys

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "src" / "calibration" / "station_stacking.py").exists():
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise RuntimeError("Could not find the weather-research project root")
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
os.environ["WEATHER_RESEARCH_INCLUDE_DIRECT_NBM"] = "1"

STATION_ID = "KATL"
FAST_MODE = False
OPTUNA_TRIALS = 30
OPTUNA_STARTUP_TRIALS = 15
STACK_OPTUNA_TRIALS = 30
STACK_OPTUNA_STARTUP_TRIALS = 15
MISSINGNESS_LIMIT = 0.03
MONTHLY_SHRINKAGE = 60.0
ROOT_OUTPUT = PROJECT_ROOT / "data" / "calibration" / "station_stacking_v19_patched" / STATION_ID
MODEL_OUTPUT = ROOT_OUTPUT / "dense_backbone"
METHOD_OUTPUT = ROOT_OUTPUT / "v19_b"
METHOD_OUTPUT.mkdir(parents=True, exist_ok=True)
PROJECT_ROOT


In [ ]:
import pandas as pd

from src.calibration.station_stacking import (
    TARGET_SOURCE_SETTLEMENT_FIRST, StationStackingConfig, YearSplitFold,
    _modeling_frame, run_station_year_split_experiment,
)
from src.calibration.v19_bucket import (
    bucket_decision_metrics, crossfit_ridge_predictions,
    empirical_modal_bucket_decisions, feature_missingness_audit,
    paired_bootstrap_bucket_gain,
)


## Forward contract

Every fold selects features from its training rows only. The final 2026 model selects from 2021–2025; neither feature selection nor residual calibration can inspect 2026 outcomes.


In [ ]:
FOLDS = (
    YearSplitFold("train_2021_valid_2022", 2021, 2021, 2022),
    YearSplitFold("train_2021_2022_valid_2023", 2021, 2022, 2023),
    YearSplitFold("train_2021_2023_valid_2024", 2021, 2023, 2024),
    YearSplitFold("train_2021_2024_valid_2025", 2021, 2024, 2025),
)
config = StationStackingConfig(
    station_id=STATION_ID, project_root=PROJECT_ROOT,
    timing_mode="same_day_11am_live_safe", providers=("gfs", "hrrr", "nbm"),
    fast_mode=FAST_MODE, optuna_trials=OPTUNA_TRIALS, stack_optuna_trials=STACK_OPTUNA_TRIALS,
    optuna_startup_trials=OPTUNA_STARTUP_TRIALS, stack_optuna_startup_trials=STACK_OPTUNA_STARTUP_TRIALS, optuna_metric="mae_f",
    optuna_verbose=True, feature_version="v11", target_mode="remaining_warmup",
    target_source=TARGET_SOURCE_SETTLEMENT_FIRST, hyperparameter_space="wide",
    base_model_methods=("xgboost", "lightgbm", "catboost"), stack_enabled=True,
    year_split_folds=FOLDS, year_split_validation_weights={2022: 1, 2023: 1, 2024: 1, 2025: 1},
    year_split_test_train_years=(2021, 2025), year_split_test_year=2026,
    max_feature_missing_fraction=MISSINGNESS_LIMIT, output_dir=MODEL_OUTPUT,
)
config


## Train the dense V19 backbone


In [ ]:
result = run_station_year_split_experiment(config)
result.scoreboard


## Audit the 3% gate


In [ ]:
modeling_frame, categorical, numeric = _modeling_frame(result.features, config)
missingness = feature_missingness_audit(
    modeling_frame, categorical, numeric, train_years=(2021, 2025),
    max_missing_fraction=MISSINGNESS_LIMIT,
)
missingness.to_csv(METHOD_OUTPUT / "feature_missingness_audit.csv", index=False)
missingness.groupby(["kind", "keep_v19"]).size().rename("feature_count").reset_index()


## Cross-fit ridge residuals


In [ ]:
residuals = crossfit_ridge_predictions(
    result.validation_predictions,
    base_model_methods=tuple(config.effective_base_model_methods),
    providers=tuple(config.providers), min_train_rows=config.effective_min_meta_train_rows,
)
residuals.to_csv(METHOD_OUTPUT / "crossfit_ridge_residuals.csv", index=False)
residuals.groupby("validation_year")["residual_f"].agg(["count", "mean", "std", "median"])


## V19-B empirical modal decision

The continuous ridge temperature is preserved. Bucket probabilities come only from cross-fitted historical residuals, with monthly estimates shrunk toward the station-global distribution.


In [ ]:
decisions = empirical_modal_bucket_decisions(
    result.test_predictions, residuals, monthly_shrinkage=MONTHLY_SHRINKAGE,
)
metrics = bucket_decision_metrics(decisions)
paired_gain = paired_bootstrap_bucket_gain(decisions)
decisions.to_csv(METHOD_OUTPUT / "2026_empirical_modal_decisions.csv", index=False)
metrics.to_csv(METHOD_OUTPUT / "2026_metrics.csv", index=False)
paired_gain.to_frame("value").to_csv(METHOD_OUTPUT / "paired_bootstrap_gain.csv")
display(metrics)
display(paired_gain.to_frame("value"))


## Interpretation

The paired gain compares V19-B's empirical modal bucket with the same dense ridge forecast rounded directly, on identical 2026 dates. Run every station before considering promotion.
